# Four weeks of negative results, and the finding I had to take back

### Playground Series S6E8, smartphone addiction

Best cross-validation **0.967665**, best public LB **0.96897**. Twenty-four logged
experiments, of which four are nulls or negatives that are written up here at the
same length as the wins.

This is not a solution notebook. Nothing in it will move you up the leaderboard by
itself. It is a record of which ideas worked, which did not, and how each one was
killed cheaply enough that the next one still had time to be tried.

**The short version.** Fifteen experiments of model capacity and stochastic
averaging were worth **+0.0089 AUC** between them, and 80% of that was noticing the untuned
baseline was underfit. Not one engineered feature helped, and neither did a second
gradient-boosting library. Then one change of *representation*, target encoding every
column, was worth **+0.0029** on its own, the second largest step in the whole
competition, and it arrived after I had already written that the board was closed.

Five things in here that I have not seen written up elsewhere for this dataset:

1. **Rank correlation between two models does not predict whether averaging them
   helps.** Measured across all 66 pairs of models I trained. This is the opposite of
   the usual advice, and it nearly cost me a day of CatBoost runs.
2. **I then over-claimed that finding, and the correction is the most useful thing
   here.** Every test behind it used an equal-weight combiner. Hold the eighteen
   models fixed and change only the combiner: averaged, they score **0.001554 below**
   the best single model; weighted by a logistic regression, **0.000908 above** it.
   The model I had rejected as the worst blend partner I ever measured takes the
   seventh largest weight of eighteen.
3. **Fold standard deviation is the wrong bar for a paired comparison** and it almost
   buried my one real improvement.
4. **The public leaderboard here has a standard error near 0.001**, larger than every
   gain after the fourth experiment. Two of my submissions demonstrate this cleanly.
5. **A verified 11-point effect in the data that is worth exactly nothing in the
   model.** The last idea I tried, and the cleanest example I have of why effect size
   and incremental value are different quantities.

The full experiment ledger, every notebook, and the working log are in the repo
linked at the bottom.

## Setup

The measured results below are inlined as data. They come from the out-of-fold
prediction vectors saved by the training runs in the repo, not from numbers typed
in by hand. The script that produced this blob recomputes every figure from those
vectors, so a number in a chart and a number in the ledger cannot drift apart.

One of the LightGBM rows is retrained live further down, so you can watch a
ledger row reproduce instead of taking the table on trust.

In [ ]:
import json

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

# Measured from the saved out-of-fold vectors. See the repo link at the bottom;
# `writeup_numbers.py` regenerates this blob from `artifacts/oof/`.
D = json.loads(r'''{"missing_base_rate":0.7094243168830872,"missingness":[{"feature":"gender","n_missing":29034,"miss_frac":0.04199494047317713,"rate":0.7087897062301636,"rate_present":0.7094521522521973,"lift":-0.0006624460220336914,"z":-0.24333532780246742},{"feature":"stress_level","n_missing":55148,"miss_frac":0.07976637656591487,"rate":0.7090012431144714,"rate_present":0.7094610333442688,"lift":-0.0004597902297973633,"z":-0.22813451817273367},{"feature":"social_media_hours","n_missing":133995,"miss_frac":0.19381111967704656,"rate":0.709414541721344,"rate_present":0.7094267010688782,"lift":-1.2159347534179688e-05,"z":-0.008802181122276542},{"feature":"academic_work_impact","n_missing":44224,"miss_frac":0.06396584168512039,"rate":0.7094563841819763,"rate_present":0.7094221711158752,"lift":3.421306610107422e-05,"z":0.015331483743799402},{"feature":"notifications_per_day","n_missing":67584,"miss_frac":0.0977538767286355,"rate":0.7102420926094055,"rate_present":0.7093357443809509,"lift":0.0009063482284545898,"z":0.4929431994854286},{"feature":"work_study_hours","n_missing":51518,"miss_frac":0.07451592420256042,"rate":0.7106448411941528,"rate_present":0.7093260884284973,"lift":0.0013187527656555176,"z":0.634226942111851},{"feature":"gaming_hours","n_missing":126821,"miss_frac":0.18343460583277527,"rate":0.7105053663253784,"rate_present":0.7091814875602722,"lift":0.0013238787651062012,"z":0.9383321982774018},{"feature":"weekend_screen_time","n_missing":112063,"miss_frac":0.16208855184423948,"rate":0.7106538414955139,"rate_present":0.7091864943504333,"lift":0.0014673471450805664,"z":0.9903310293760929},{"feature":"daily_screen_time_hours","n_missing":95854,"miss_frac":0.13864376331597164,"rate":0.7113004922866821,"rate_present":0.709122359752655,"lift":0.0021781325340270996,"z":1.3784726525679663},{"feature":"app_opens_per_day","n_missing":80710,"miss_frac":0.1167393967620764,"rate":0.7127864956855774,"rate_present":0.7089799642562866,"lift":0.0038065314292907715,"z":2.2384884320856524},{"feature":"age","n_missing":28929,"miss_frac":0.04184306788415448,"rate":0.7134017944335938,"rate_present":0.7092506289482117,"lift":0.00415116548538208,"z":1.5222024319584473},{"feature":"sleep_hours","n_missing":44480,"miss_frac":0.06433612152121371,"rate":0.7133768200874329,"rate_present":0.7091525793075562,"lift":0.004224240779876709,"z":1.8980529200639886}],"single_cv":{"anchor (100 trees)":{"mean":0.9549467010256301,"sd":0.00064490396666483},"300 trees":{"mean":0.9606048423752365,"sd":0.0006875902743794306},"1000 trees":{"mean":0.9621409051321341,"sd":0.0008590300603214384},"2000 trees":{"mean":0.9618323239022759,"sd":0.000952472720968497},"lr 0.10":{"mean":0.9621982367135239,"sd":0.0008159296161059919},"lr 0.05":{"mean":0.9632098272187225,"sd":0.000591461409111401},"lr 0.03":{"mean":0.9632745391279285,"sd":0.0005493987703548775},"bagged seed 42":{"mean":0.9634705327026813,"sd":0.0005907357588788866},"bagged seed 2024":{"mean":0.9632336420315293,"sd":0.0008994906856684058},"bagged seed 7":{"mean":0.963445428051234,"sd":0.0004776115887212625},"bagged seed 2025":{"mean":0.9633374484598652,"sd":0.0007310625216525802},"bagged seed 13":{"mean":0.9634827033576115,"sd":0.0005519140335981485}},"pairs":[{"a":"anchor (100 trees)","b":"300 trees","spearman":0.9930765017385362,"cv_a":0.9549467010256301,"cv_b":0.9606048423752365,"cv_gap":0.005658141349606405,"blend":0.9584202404520668,"gain":-0.0021846019231697156,"paired_sd":0.00014357766590540974,"folds_won":0},{"a":"anchor (100 trees)","b":"1000 trees","spearman":0.9836829150135086,"cv_a":0.9549467010256301,"cv_b":0.9621409051321341,"cv_gap":0.007194204106504065,"blend":0.9600452697116528,"gain":-0.002095635420481301,"paired_sd":0.0002671712392164481,"folds_won":0},{"a":"anchor (100 trees)","b":"2000 trees","spearman":0.9745648778947578,"cv_a":0.9549467010256301,"cv_b":0.9618323239022759,"cv_gap":0.006885622876645847,"blend":0.9602860398817803,"gain":-0.0015462840204955696,"paired_sd":0.00014914676593473817,"folds_won":0},{"a":"anchor (100 trees)","b":"lr 0.10","spearman":0.984439257835441,"cv_a":0.9549467010256301,"cv_b":0.9621982367135239,"cv_gap":0.007251535687893829,"blend":0.960145659778119,"gain":-0.0020525769354049483,"paired_sd":0.00027167713997864815,"folds_won":0},{"a":"anchor (100 trees)","b":"lr 0.05","spearman":0.9890902353771174,"cv_a":0.9549467010256301,"cv_b":0.9632098272187225,"cv_gap":0.00826312619309244,"blend":0.96043968609283,"gain":-0.002770141125892489,"paired_sd":8.846259614537647e-05,"folds_won":0},{"a":"anchor (100 trees)","b":"lr 0.03","spearman":0.9888691181011995,"cv_a":0.9549467010256301,"cv_b":0.9632745391279285,"cv_gap":0.008327838102298424,"blend":0.9604384451434804,"gain":-0.0028360939844481294,"paired_sd":7.756749282176583e-05,"folds_won":0},{"a":"anchor (100 trees)","b":"bagged seed 42","spearman":0.9883336427547597,"cv_a":0.9549467010256301,"cv_b":0.9634705327026813,"cv_gap":0.008523831677051286,"blend":0.9606445874293545,"gain":-0.0028259452733267352,"paired_sd":6.278914037092448e-05,"folds_won":0},{"a":"anchor (100 trees)","b":"bagged seed 2024","spearman":0.9834789992708939,"cv_a":0.9549467010256301,"cv_b":0.9632336420315293,"cv_gap":0.008286941005899218,"blend":0.9606286456529392,"gain":-0.0026049963785901966,"paired_sd":0.0004952931952778384,"folds_won":0},{"a":"anchor (100 trees)","b":"bagged seed 7","spearman":0.9870274795339734,"cv_a":0.9549467010256301,"cv_b":0.963445428051234,"cv_gap":0.008498727025603947,"blend":0.9607032757016605,"gain":-0.0027421523495735345,"paired_sd":0.00019378306930941934,"folds_won":0},{"a":"anchor (100 trees)","b":"bagged seed 2025","spearman":0.9856464416771401,"cv_a":0.9549467010256301,"cv_b":0.9633374484598652,"cv_gap":0.00839074743423518,"blend":0.9606608259490772,"gain":-0.0026766225107882403,"paired_sd":0.0003754351886575923,"folds_won":0},{"a":"anchor (100 trees)","b":"bagged seed 13","spearman":0.9876338741521341,"cv_a":0.9549467010256301,"cv_b":0.9634827033576115,"cv_gap":0.008536002331981485,"blend":0.9606766054730324,"gain":-0.0028060978845790173,"paired_sd":8.744022263481383e-05,"folds_won":0},{"a":"300 trees","b":"1000 trees","spearman":0.9885054227814063,"cv_a":0.9606048423752365,"cv_b":0.9621409051321341,"cv_gap":0.0015360627568976604,"blend":0.9619497164809061,"gain":-0.00019118865122798035,"paired_sd":0.0003105898197343192,"folds_won":1},{"a":"300 trees","b":"2000 trees","spearman":0.9787488030888335,"cv_a":0.9606048423752365,"cv_b":0.9618323239022759,"cv_gap":0.0012274815270394424,"blend":0.9621155630162603,"gain":0.0002832391139844681,"paired_sd":0.00025752330111137597,"folds_won":5},{"a":"300 trees","b":"lr 0.10","spearman":0.9897116101269284,"cv_a":0.9606048423752365,"cv_b":0.9621982367135239,"cv_gap":0.0015933943382874238,"blend":0.9620302270499028,"gain":-0.00016800966362118253,"paired_sd":0.00032083934708392205,"folds_won":1},{"a":"300 trees","b":"lr 0.05","spearman":0.9928298718287766,"cv_a":0.9606048423752365,"cv_b":0.9632098272187225,"cv_gap":0.0026049848434860357,"blend":0.9623782978541178,"gain":-0.0008315293646048883,"paired_sd":3.146865347652024e-05,"folds_won":0},{"a":"300 trees","b":"lr 0.03","spearman":0.9925160458419798,"cv_a":0.9606048423752365,"cv_b":0.9632745391279285,"cv_gap":0.002669696752692019,"blend":0.9623837563003457,"gain":-0.0008907828275825702,"paired_sd":5.81414587942412e-05,"folds_won":0},{"a":"300 trees","b":"bagged seed 42","spearman":0.9917988890272705,"cv_a":0.9606048423752365,"cv_b":0.9634705327026813,"cv_gap":0.002865690327444881,"blend":0.962573015847077,"gain":-0.0008975168556043967,"paired_sd":3.177181594536487e-05,"folds_won":0},{"a":"300 trees","b":"bagged seed 2024","spearman":0.9875189229544078,"cv_a":0.9606048423752365,"cv_b":0.9632336420315293,"cv_gap":0.002628799656292813,"blend":0.9625809392396649,"gain":-0.0006527027918643569,"paired_sd":0.0005196698959231307,"folds_won":1},{"a":"300 trees","b":"bagged seed 7","spearman":0.9910458358371231,"cv_a":0.9606048423752365,"cv_b":0.963445428051234,"cv_gap":0.0028405856759975423,"blend":0.9626551901841743,"gain":-0.0007902378670594957,"paired_sd":0.00016339046508889087,"folds_won":0},{"a":"300 trees","b":"bagged seed 2025","spearman":0.9896924463922334,"cv_a":0.9606048423752365,"cv_b":0.9633374484598652,"cv_gap":0.0027326060846287747,"blend":0.9626115939392029,"gain":-0.0007258545206625833,"paired_sd":0.00036815509459561774,"folds_won":0},{"a":"300 trees","b":"bagged seed 13","spearman":0.9915065958175602,"cv_a":0.9606048423752365,"cv_b":0.9634827033576115,"cv_gap":0.00287786098237508,"blend":0.9626346921274094,"gain":-0.000848011230202217,"paired_sd":5.133159390636951e-05,"folds_won":0},{"a":"1000 trees","b":"2000 trees","spearman":0.9838366777610164,"cv_a":0.9621409051321341,"cv_b":0.9618323239022759,"cv_gap":0.000308581229858218,"blend":0.9626444131158106,"gain":0.0005035079836765322,"paired_sd":0.0003067286024402627,"folds_won":5},{"a":"1000 trees","b":"lr 0.10","spearman":0.993022042353452,"cv_a":0.9621409051321341,"cv_b":0.9621982367135239,"cv_gap":5.7331581389763464e-05,"blend":0.9625869982602765,"gain":0.00038876154675240306,"paired_sd":0.00037827810147254804,"folds_won":4},{"a":"1000 trees","b":"lr 0.05","spearman":0.988195036539881,"cv_a":0.9621409051321341,"cv_b":0.9632098272187225,"cv_gap":0.0010689220865883753,"blend":0.9631437658770949,"gain":-6.606134162769894e-05,"paired_sd":0.00011184498479289316,"folds_won":1},{"a":"1000 trees","b":"lr 0.03","spearman":0.9872628803978186,"cv_a":0.9621409051321341,"cv_b":0.9632745391279285,"cv_gap":0.0011336339957943586,"blend":0.9631526335033342,"gain":-0.00012190562459422072,"paired_sd":0.00010329971947249546,"folds_won":1},{"a":"1000 trees","b":"bagged seed 42","spearman":0.986384154208615,"cv_a":0.9621409051321341,"cv_b":0.9634705327026813,"cv_gap":0.0013296275705472205,"blend":0.96335338176835,"gain":-0.00011715093433148916,"paired_sd":9.469678664213163e-05,"folds_won":1},{"a":"1000 trees","b":"bagged seed 2024","spearman":0.9857334506794263,"cv_a":0.9621409051321341,"cv_b":0.9632336420315293,"cv_gap":0.0010927368993951525,"blend":0.9633854237034913,"gain":0.00015178167196217007,"paired_sd":0.0005793157968198004,"folds_won":2},{"a":"1000 trees","b":"bagged seed 7","spearman":0.9855514711213312,"cv_a":0.9621409051321341,"cv_b":0.963445428051234,"cv_gap":0.001304522919099882,"blend":0.9634484920717767,"gain":3.0640205427756586e-06,"paired_sd":0.0002697152769427477,"folds_won":2},{"a":"1000 trees","b":"bagged seed 2025","spearman":0.9862711547093388,"cv_a":0.9621409051321341,"cv_b":0.9633374484598652,"cv_gap":0.0011965433277311144,"blend":0.9634056263858917,"gain":6.81779260263582e-05,"paired_sd":0.00039237778156615686,"folds_won":2},{"a":"1000 trees","b":"bagged seed 13","spearman":0.98603856433902,"cv_a":0.9621409051321341,"cv_b":0.9634827033576115,"cv_gap":0.0013417982254774197,"blend":0.963429531490062,"gain":-5.317186754929537e-05,"paired_sd":0.00012923666079599753,"folds_won":1},{"a":"2000 trees","b":"lr 0.10","spearman":0.9833150022204985,"cv_a":0.9618323239022759,"cv_b":0.9621982367135239,"cv_gap":0.0003659128112479815,"blend":0.9626918518433328,"gain":0.0004936151298087887,"paired_sd":0.00031259624859980735,"folds_won":5},{"a":"2000 trees","b":"lr 0.05","spearman":0.979252069284783,"cv_a":0.9618323239022759,"cv_b":0.9632098272187225,"cv_gap":0.0013775033164465933,"blend":0.9631591077331179,"gain":-5.071948560479989e-05,"paired_sd":0.00018701065513062178,"folds_won":2},{"a":"2000 trees","b":"lr 0.03","spearman":0.9784559497000986,"cv_a":0.9618323239022759,"cv_b":0.9632745391279285,"cv_gap":0.0014422152256525766,"blend":0.9631730297189047,"gain":-0.00010150940902369232,"paired_sd":0.00018153760995878206,"folds_won":2},{"a":"2000 trees","b":"bagged seed 42","spearman":0.9778607195685959,"cv_a":0.9618323239022759,"cv_b":0.9634705327026813,"cv_gap":0.0016382088004054385,"blend":0.9633665969708053,"gain":-0.00010393573187614802,"paired_sd":0.0001713721615862099,"folds_won":2},{"a":"2000 trees","b":"bagged seed 2024","spearman":0.9774052957938391,"cv_a":0.9618323239022759,"cv_b":0.9632336420315293,"cv_gap":0.0014013181292533705,"blend":0.9634046403472538,"gain":0.00017099831572446876,"paired_sd":0.0006024839590794951,"folds_won":3},{"a":"2000 trees","b":"bagged seed 7","spearman":0.9772698905921975,"cv_a":0.9618323239022759,"cv_b":0.963445428051234,"cv_gap":0.0016131041489581,"blend":0.9634685934347784,"gain":2.3165383544276884e-05,"paired_sd":0.00032083577357150725,"folds_won":2},{"a":"2000 trees","b":"bagged seed 2025","spearman":0.9773175578834753,"cv_a":0.9618323239022759,"cv_b":0.9633374484598652,"cv_gap":0.0015051245575893324,"blend":0.9634248963271659,"gain":8.744786730061627e-05,"paired_sd":0.0004201743134994768,"folds_won":3},{"a":"2000 trees","b":"bagged seed 13","spearman":0.9776912226772052,"cv_a":0.9618323239022759,"cv_b":0.9634827033576115,"cv_gap":0.0016503794553356377,"blend":0.963449722416399,"gain":-3.298094121246819e-05,"paired_sd":0.00019313218884922797,"folds_won":2},{"a":"lr 0.10","b":"lr 0.05","spearman":0.9892230127072061,"cv_a":0.9621982367135239,"cv_b":0.9632098272187225,"cv_gap":0.0010115905051986118,"blend":0.9631919295609059,"gain":-1.789765781663455e-05,"paired_sd":7.606636672390308e-05,"folds_won":1},{"a":"lr 0.10","b":"lr 0.03","spearman":0.9880457011882899,"cv_a":0.9621982367135239,"cv_b":0.9632745391279285,"cv_gap":0.0010763024144045952,"blend":0.9632010889645475,"gain":-7.345016338087263e-05,"paired_sd":7.389008662888954e-05,"folds_won":1},{"a":"lr 0.10","b":"bagged seed 42","spearman":0.9875360821450077,"cv_a":0.9621982367135239,"cv_b":0.9634705327026813,"cv_gap":0.001272295989157457,"blend":0.9634040903219374,"gain":-6.644238074391407e-05,"paired_sd":7.295835529471971e-05,"folds_won":1},{"a":"lr 0.10","b":"bagged seed 2024","spearman":0.9869777426626942,"cv_a":0.9621982367135239,"cv_b":0.9632336420315293,"cv_gap":0.001035405318005389,"blend":0.963437960593061,"gain":0.00020431856153169115,"paired_sd":0.0005174098791223362,"folds_won":3},{"a":"lr 0.10","b":"bagged seed 7","spearman":0.9865373605534884,"cv_a":0.9621982367135239,"cv_b":0.963445428051234,"cv_gap":0.0012471913377101185,"blend":0.9635004824253564,"gain":5.5054374122454064e-05,"paired_sd":0.00023473099132235637,"folds_won":3},{"a":"lr 0.10","b":"bagged seed 2025","spearman":0.9873094081837185,"cv_a":0.9621982367135239,"cv_b":0.9633374484598652,"cv_gap":0.001139211746341351,"blend":0.9634579694930135,"gain":0.00012052103314810214,"paired_sd":0.0003227881298622006,"folds_won":2},{"a":"lr 0.10","b":"bagged seed 13","spearman":0.9872940895721637,"cv_a":0.9621982367135239,"cv_b":0.9634827033576115,"cv_gap":0.0012844666440876562,"blend":0.9634802727457139,"gain":-2.4306118975081502e-06,"paired_sd":9.735444608424457e-05,"folds_won":2},{"a":"lr 0.05","b":"lr 0.03","spearman":0.998091892385352,"cv_a":0.9632098272187225,"cv_b":0.9632745391279285,"cv_gap":6.471190920598335e-05,"blend":0.9633468399038323,"gain":7.230077590392181e-05,"paired_sd":4.177970762279755e-05,"folds_won":5},{"a":"lr 0.05","b":"bagged seed 42","spearman":0.9957503870292943,"cv_a":0.9632098272187225,"cv_b":0.9634705327026813,"cv_gap":0.0002607054839588452,"blend":0.9635728739572048,"gain":0.00010234125452337483,"paired_sd":2.0210071213462815e-05,"folds_won":5},{"a":"lr 0.05","b":"bagged seed 2024","spearman":0.9913523243456657,"cv_a":0.9632098272187225,"cv_b":0.9632336420315293,"cv_gap":2.381481280677722e-05,"blend":0.963601173661026,"gain":0.00036753162949680894,"paired_sd":0.0005356904441392467,"folds_won":5},{"a":"lr 0.05","b":"bagged seed 7","spearman":0.994866027966549,"cv_a":0.9632098272187225,"cv_b":0.963445428051234,"cv_gap":0.00023560083251150665,"blend":0.963670168938625,"gain":0.00022474088739101727,"paired_sd":0.0001651926687614814,"folds_won":5},{"a":"lr 0.05","b":"bagged seed 2025","spearman":0.9931959904996398,"cv_a":0.9632098272187225,"cv_b":0.9633374484598652,"cv_gap":0.0001276212411427391,"blend":0.9636267335120678,"gain":0.00028928505220238154,"paired_sd":0.00035726285824102155,"folds_won":5},{"a":"lr 0.05","b":"bagged seed 13","spearman":0.9947631273975702,"cv_a":0.9632098272187225,"cv_b":0.9634827033576115,"cv_gap":0.0002728761388890444,"blend":0.9636543963713378,"gain":0.00017169301372617075,"paired_sd":3.172005260813579e-05,"folds_won":5},{"a":"lr 0.03","b":"bagged seed 42","spearman":0.9961549651143419,"cv_a":0.9632745391279285,"cv_b":0.9634705327026813,"cv_gap":0.00019599357475286183,"blend":0.9635807338766404,"gain":0.00011020117395905693,"paired_sd":3.9165531996792825e-05,"folds_won":5},{"a":"lr 0.03","b":"bagged seed 2024","spearman":0.990550712129159,"cv_a":0.9632745391279285,"cv_b":0.9632336420315293,"cv_gap":4.0897096399206134e-05,"blend":0.9636112997552413,"gain":0.00033676062731287094,"paired_sd":0.00014561005900725176,"folds_won":5},{"a":"lr 0.03","b":"bagged seed 7","spearman":0.9947482610126726,"cv_a":0.9632745391279285,"cv_b":0.963445428051234,"cv_gap":0.0001708889233055233,"blend":0.9636787049227694,"gain":0.00023327687153544828,"paired_sd":0.00017757322392750798,"folds_won":5},{"a":"lr 0.03","b":"bagged seed 2025","spearman":0.992860946772584,"cv_a":0.9632745391279285,"cv_b":0.9633374484598652,"cv_gap":6.290933193675574e-05,"blend":0.9636357347513217,"gain":0.0002982862914564066,"paired_sd":0.0003638300855812459,"folds_won":5},{"a":"lr 0.03","b":"bagged seed 13","spearman":0.9948778894230152,"cv_a":0.9632745391279285,"cv_b":0.9634827033576115,"cv_gap":0.00020816422968306103,"blend":0.9636622300383962,"gain":0.00017952668078475843,"paired_sd":3.7339744282254896e-05,"folds_won":5},{"a":"bagged seed 42","b":"bagged seed 2024","spearman":0.989781705992632,"cv_a":0.9634705327026813,"cv_b":0.9632336420315293,"cv_gap":0.00023689067115206797,"blend":0.9637088284874322,"gain":0.00023829578475076385,"paired_sd":0.00014183887570779546,"folds_won":5},{"a":"bagged seed 42","b":"bagged seed 7","spearman":0.9952482999788735,"cv_a":0.9634705327026813,"cv_b":0.963445428051234,"cv_gap":2.5104651447338533e-05,"blend":0.9637699273723278,"gain":0.0002993946696464134,"paired_sd":7.607100152180388e-05,"folds_won":5},{"a":"bagged seed 42","b":"bagged seed 2025","spearman":0.9925200396846862,"cv_a":0.9634705327026813,"cv_b":0.9633374484598652,"cv_gap":0.0001330842428161061,"blend":0.9637342938203286,"gain":0.0002637611176472099,"paired_sd":6.152003333392871e-05,"folds_won":5},{"a":"bagged seed 42","b":"bagged seed 13","spearman":0.9963487474204312,"cv_a":0.9634705327026813,"cv_b":0.9634827033576115,"cv_gap":1.2170654930199198e-05,"blend":0.9637588004267232,"gain":0.00027609706911186916,"paired_sd":4.807158876957789e-05,"folds_won":5},{"a":"bagged seed 2024","b":"bagged seed 7","spearman":0.9902793175540823,"cv_a":0.9632336420315293,"cv_b":0.963445428051234,"cv_gap":0.00021178601970472943,"blend":0.9637195903091171,"gain":0.0002741622578831704,"paired_sd":0.00022970901337129355,"folds_won":4},{"a":"bagged seed 2024","b":"bagged seed 2025","spearman":0.992782079445031,"cv_a":0.9632336420315293,"cv_b":0.9633374484598652,"cv_gap":0.00010380642833596188,"blend":0.9636978171807534,"gain":0.00036036872088816006,"paired_sd":0.0002568694265550775,"folds_won":5},{"a":"bagged seed 2024","b":"bagged seed 13","spearman":0.9899107954603206,"cv_a":0.9632336420315293,"cv_b":0.9634827033576115,"cv_gap":0.00024906132608226716,"blend":0.9637077852933901,"gain":0.0002250819357786149,"paired_sd":0.0001353015392851377,"folds_won":4},{"a":"bagged seed 7","b":"bagged seed 2025","spearman":0.9920436551301872,"cv_a":0.963445428051234,"cv_b":0.9633374484598652,"cv_gap":0.00010797959136876756,"blend":0.9637548888019702,"gain":0.0003094607507363234,"paired_sd":0.00016682899948468972,"folds_won":5},{"a":"bagged seed 7","b":"bagged seed 13","spearman":0.9955129368789487,"cv_a":0.963445428051234,"cv_b":0.9634827033576115,"cv_gap":3.727530637753773e-05,"blend":0.9637456075863582,"gain":0.0002629042287467298,"paired_sd":5.067585379873403e-05,"folds_won":5},{"a":"bagged seed 2025","b":"bagged seed 13","spearman":0.9926527702006471,"cv_a":0.9633374484598652,"cv_b":0.9634827033576115,"cv_gap":0.0001452548977463053,"blend":0.9637338506377544,"gain":0.00025114728014290487,"paired_sd":4.913150107965401e-05,"folds_won":5}],"seed_saturation":[{"n_seeds":1,"cv":0.9634705327026813,"sd":0.0005907357588788866},{"n_seeds":2,"cv":0.9637088284874322,"sd":0.0006021893977001631},{"n_seeds":3,"cv":0.9638210471372626,"sd":0.0005601863998334479},{"n_seeds":4,"cv":0.9638575079829785,"sd":0.0005564493204720445},{"n_seeds":5,"cv":0.9638804130611953,"sd":0.0005547070146738952}],"neural":{"cv":0.9391689164198243,"sd":0.000759146589520503,"lgbm_blend_cv":0.9638804130611953,"spearman_vs_blend":0.9650419155042297,"within_family_spearman_min":0.9745648778947578,"within_family_spearman_max":0.998091892385352,"catboost_spearman":0.9877,"weight_curve":[{"w":0.05,"blend":0.9636925088678598,"gain":-0.00018790419333551965,"paired_sd":1.584787372166672e-05,"folds_won":0},{"w":0.1,"blend":0.9634012345881462,"gain":-0.00047917847304914665,"paired_sd":3.266236322796298e-05,"folds_won":0},{"w":0.15,"blend":0.9630017114431585,"gain":-0.0008787016180368257,"paired_sd":5.0449512012949546e-05,"folds_won":0},{"w":0.2,"blend":0.9624893155070259,"gain":-0.0013910975541695514,"paired_sd":6.803900177538933e-05,"folds_won":0},{"w":0.25,"blend":0.9618601560109207,"gain":-0.002020257050274665,"paired_sd":8.61945047688188e-05,"folds_won":0},{"w":0.3,"blend":0.9611110810618296,"gain":-0.0027693319993658204,"paired_sd":0.0001051837996202618,"folds_won":0},{"w":0.4,"blend":0.959244330946718,"gain":-0.004636082114477369,"paired_sd":0.00014656530985861184,"folds_won":0},{"w":0.5,"blend":0.9568859117407996,"gain":-0.006994501320395918,"paired_sd":0.0001917106564771205,"folds_won":0}]},"target_encoding":{"cv":0.9667823810724869,"sd":0.00045339845274297596,"vs_same_model_raw":0.003311848369805359,"paired_sd":0.00027030808391996584,"folds_won":5,"seed_cv":{"te42":0.9667823810724869,"te2024":0.9667707371023573,"te7":0.9667285893376194,"te2025":0.9667433669355111,"te13":0.9667894194900226}},"combiners":[{"name":"best single model","auc":0.9668002879455457,"gain":0.0,"paired_sd":0.0,"splits_won":0},{"name":"rank mean, 5 seeds","auc":0.9672337289639643,"gain":0.0004334410184185566,"paired_sd":1.0266049366342973e-05,"splits_won":5},{"name":"rank mean, all 18","auc":0.9652460727650588,"gain":-0.0015542151804869286,"paired_sd":5.8776663895356795e-05,"splits_won":0},{"name":"logit stack, 5 seeds","auc":0.9672318074586792,"gain":0.0004315195131335825,"paired_sd":9.806374427543902e-06,"splits_won":5},{"name":"logit stack, all 18","auc":0.9677082743259209,"gain":0.0009079863803753918,"paired_sd":1.803318377621806e-05,"splits_won":5},{"name":"logit stack, 17 no neural","auc":0.9676152018717368,"gain":0.0008149139261911298,"paired_sd":1.6729749928232447e-05,"splits_won":5},{"name":"rank mean, best + neural","auc":0.9605283637839381,"gain":-0.006271924161607578,"paired_sd":7.080080688970952e-05,"splits_won":0},{"name":"logit stack, best + neural","auc":0.9668905655408114,"gain":9.027759526580859e-05,"paired_sd":6.80090222539322e-06,"splits_won":5}],"neural_in_stack":{"gain":9.307245418426202e-05,"paired_sd":3.3165068998384373e-06,"splits_won":5},"stack_coefs":[{"name":"te13","cv":0.9667894194900226,"coef":0.1661686536700836},{"name":"te42","cv":0.9667823810724869,"coef":0.1698624269180501},{"name":"te2024","cv":0.9667707371023573,"coef":0.1643876968929748},{"name":"te2025","cv":0.9667433669355111,"coef":0.1550868561135781},{"name":"te7","cv":0.9667285893376194,"coef":0.13707896946315176},{"name":"bagged_seed_13","cv":0.9634827033576115,"coef":0.08513170605223949},{"name":"bagged_seed_42","cv":0.9634705327026813,"coef":0.09490139210354506},{"name":"bagged_seed_7","cv":0.963445428051234,"coef":0.09061994698070062},{"name":"bagged_seed_2025","cv":0.9633374484598652,"coef":0.06822157570398021},{"name":"lr_0.03","cv":0.9632745391279285,"coef":0.1302179659154745},{"name":"bagged_seed_2024","cv":0.9632336420315293,"coef":0.010133192471048148},{"name":"lr_0.05","cv":0.9632098272187225,"coef":0.10965562107933086},{"name":"lr_0.10","cv":0.9621982367135239,"coef":-0.01952483566725401},{"name":"1000_trees","cv":0.9621409051321341,"coef":-0.009580320659257933},{"name":"2000_trees","cv":0.9618323239022759,"coef":-0.011818217381742786},{"name":"300_trees","cv":0.9606048423752365,"coef":-0.13210997492586757},{"name":"anchor_(100_trees)","cv":0.9549467010256301,"coef":-0.36190176503531174},{"name":"neural","cv":0.9391689164198243,"coef":0.11775909426086155}],"ledger":[{"id":1,"name":"lgbm_default_anchor","cv_mean":0.954947,"cv_std":0.000645,"lb_public":0.95594},{"id":2,"name":"lgbm_trees100","cv_mean":0.954947,"cv_std":0.000645,"lb_public":null},{"id":3,"name":"lgbm_trees300","cv_mean":0.960605,"cv_std":0.000688,"lb_public":null},{"id":4,"name":"lgbm_trees1000","cv_mean":0.962141,"cv_std":0.000859,"lb_public":0.96435},{"id":5,"name":"lgbm_trees2000","cv_mean":0.961832,"cv_std":0.000952,"lb_public":null},{"id":6,"name":"lgbm_lr01","cv_mean":0.962198,"cv_std":0.000816,"lb_public":null},{"id":7,"name":"lgbm_lr005","cv_mean":0.96321,"cv_std":0.000591,"lb_public":null},{"id":8,"name":"lgbm_lr003","cv_mean":0.963275,"cv_std":0.000549,"lb_public":0.96478},{"id":9,"name":"lgbm_bag08_seed42","cv_mean":0.963471,"cv_std":0.000591,"lb_public":null},{"id":10,"name":"lgbm_bag08_seed2024","cv_mean":0.963234,"cv_std":0.000899,"lb_public":null},{"id":11,"name":"lgbm_bag08_seed7","cv_mean":0.963445,"cv_std":0.000478,"lb_public":null},{"id":12,"name":"lgbm_bag08_seedblend3","cv_mean":0.963821,"cv_std":0.00056,"lb_public":0.96509},{"id":13,"name":"lgbm_bag08_seed2025","cv_mean":0.963337,"cv_std":0.000731,"lb_public":null},{"id":14,"name":"lgbm_bag08_seed13","cv_mean":0.963483,"cv_std":0.000552,"lb_public":null},{"id":15,"name":"lgbm_bag08_seedblend5","cv_mean":0.96388,"cv_std":0.000555,"lb_public":0.96508},{"id":16,"name":"neural_mlp_kaggle","cv_mean":0.939169,"cv_std":0.000759,"lb_public":null},{"id":17,"name":"lgbm_bag08_seed42_te","cv_mean":0.966782,"cv_std":0.000453,"lb_public":0.96825},{"id":18,"name":"lgbm_bag08_seed42_te_imp","cv_mean":0.966805,"cv_std":0.000469,"lb_public":null},{"id":19,"name":"lgbm_bag08_seed42_te_lat","cv_mean":0.966651,"cv_std":0.000515,"lb_public":null},{"id":20,"name":"lgbm_bag08_seed2024_te","cv_mean":0.966771,"cv_std":0.000446,"lb_public":null},{"id":21,"name":"lgbm_bag08_seed7_te","cv_mean":0.966729,"cv_std":0.000433,"lb_public":null},{"id":22,"name":"lgbm_bag08_seed2025_te","cv_mean":0.966743,"cv_std":0.000427,"lb_public":null},{"id":23,"name":"lgbm_bag08_seed13_te","cv_mean":0.966789,"cv_std":0.000427,"lb_public":null},{"id":24,"name":"stack_logit_18","cv_mean":0.967665,"cv_std":0.000432,"lb_public":0.96897},{"id":25,"name":"stack_logit_18_oof","cv_mean":0.96765,"cv_std":0.000437,"lb_public":null}]}''')

print(f"{len(D['pairs'])} model pairs, {len(D['ledger'])} ledger rows, "
      f"{len(D['missingness'])} features")

### Chart styling

Two colours, checked for colour-vision deficiency separation against a white
background rather than picked by eye. Everything else is recessive on purpose:
hairline grid, no top or right spine, labels in grey so the marks carry the ink.

In [ ]:
BLUE, ORANGE = "#2a78d6", "#eb6834"
INK, SECOND, MUTED = "#0b0b0b", "#52514e", "#898781"
GRID, AXIS, SURFACE = "#e1e0d9", "#c3c2b7", "#ffffff"

plt.rcParams.update({
    "figure.facecolor": SURFACE, "axes.facecolor": SURFACE,
    "text.color": INK, "axes.labelcolor": SECOND, "axes.edgecolor": AXIS,
    "xtick.color": MUTED, "ytick.color": MUTED,
    "axes.grid": True, "grid.color": GRID, "grid.linewidth": 0.8,
    "axes.spines.top": False, "axes.spines.right": False,
    "font.size": 10, "figure.dpi": 120,
})


def finish(ax, title, sub=None):
    """Title and subtitle stacked above the axes.

    Placed by hand rather than with set_title, which collides with a subtitle
    drawn just above the axes.
    """
    n = sub.count("\n") + 1 if sub else 0
    ax.text(0, 1.06 + 0.055 * n, title, transform=ax.transAxes, color=INK,
            fontsize=12, fontweight="bold", va="bottom")
    if sub:
        ax.text(0, 1.03, sub, transform=ax.transAxes, color=SECOND,
                fontsize=9.5, va="bottom", linespacing=1.4)
    ax.set_axisbelow(True)


print("styled")

## First: the metric and the validation scheme

Before looking at a single feature. The metric is ROC AUC on a predicted
probability, which decides three things at once: the loss is log loss, the
predictions never need calibrating because AUC only reads their order, and rank
averaging is the natural way to combine models.

The split is **5-fold stratified, shuffled, seed 42**, and it never changed. There
are no groups, no repeated entities and no time index in this data, so stratifying
on the target is the whole requirement. Every number in this notebook comes from
that one split, which is what makes them comparable to each other.

The thing worth copying is not the split. It is that **the split was fixed and the
fold assignment checksummed** before any modelling. Every out-of-fold vector the
repo saved reproduces its recorded fold-mean to within 5e-7, so any two of them can
be blended and compared row for row, months apart. Half of what follows would have
been unmeasurable without that.

## The data is synthetic, and that decided the feature strategy

Playground Series data is generated from an original public dataset. Community
forensic work on this one (credit to Busya PRIME and broccoli beef in the
competition discussion) reported that the original is close to a lookup table:
`addicted` is 1 when `daily_screen_time_hours > 8` or `social_media_hours > 4`.

I did not verify those rules myself, so treat them as reported rather than
established. They score about **0.9888 AUC on the original data and about
0.835 on the synthetic data we are actually scored on.**

That single pair of numbers rewrote my plan. My feature list was a stack of
threshold and ratio features aimed squarely at recovering those rules, and an
untuned LightGBM already scores 0.9549, well above the 0.835 ceiling the rules
themselves reach here. The generator has smeared the decision boundary into
something smooth, and encoding the original rules would have been a step backwards.

**Read the discussion tab before writing features.** It cost me twenty minutes and
saved a week of the most obvious possible dead end.

## My top-priority idea, killed in ninety seconds

Every feature in this dataset has missing values, between 4% and 20%. That is a
conspicuous amount of structure, and missingness indicators were the first thing on
my list, the reasoning being that whether someone declined to report their screen
time might say more than the number itself.

It is a testable claim and it costs one pass over the data, so I tested it before
spending a training run: for each feature, is the target rate different when that
feature is missing?

In [ ]:
mi = sorted(D["missingness"], key=lambda m: m["lift"])
names = [m["feature"] for m in mi]
lift = np.array([m["lift"] for m in mi])
# Each feature's own standard error, recovered from its z. A single shared band
# would have to be drawn at the widest feature's error, which would understate the
# certainty on the other eleven and flatter the conclusion.
se = np.array([abs(m["lift"] / m["z"]) if m["z"] else np.nan for m in mi])

fig, ax = plt.subplots(figsize=(9, 5))
yy = np.arange(len(names))
ax.axvline(0, color=AXIS, lw=1.2, zorder=1)
ax.errorbar(lift, yy, xerr=2 * se, fmt="o", color=BLUE, ms=8, mec=SURFACE,
            mew=1.5, ecolor=MUTED, elinewidth=1.4, capsize=3.5, capthick=1.4,
            zorder=3, label="±2 standard errors")
ax.set_yticks(yy)
ax.set_yticklabels(names, fontsize=9.5, color=SECOND)
ax.set_xlabel("target rate when the feature is missing, minus when it is present")
ax.legend(frameon=False, loc="lower right", labelcolor=SECOND)
ax.grid(axis="y", visible=False)
finish(ax, "Missingness carries no signal about the target",
       "Eleven of twelve intervals cross zero. The largest |z| is 2.24, against the\n"
       "2.4 you would expect from pure noise across twelve tests.")
plt.tight_layout()
plt.show()

print(f"largest |z|: {max(abs(m['z']) for m in mi):.2f}")

Not one feature clears the bar. The largest |z| across twelve tests is **2.24**,
and the largest value you would *expect* from twelve draws of pure noise is about
2.4. `app_opens_per_day` is the one interval that excludes zero, which is what one
in twelve looks like when nothing is going on.

The missingness was injected at random with respect to the target. LightGBM already
routes NaN natively, so twelve indicator columns would have added twelve columns of
nothing.

**This is the part I would most like people to copy.** The idea was wrong, and being
wrong cost ninety seconds instead of a training run and a submission, because it was
phrased as a measurable claim before it was phrased as a feature.

## Adversarial validation: mild, diffuse, not actionable

Train a classifier to tell train rows from test rows. If it succeeds, the two sets
differ and your CV is measuring the wrong distribution.

It scored **AUC 0.562868 +/- 0.000792**. Real but mild, and importance was spread
almost evenly across all eight numeric features at 10-11% each. A concentrated shift
gives you a feature to drop. A diffuse one gives you nothing to act on, so I noted
it and moved on rather than reaching for fold weighting.

Worth running anyway: it is ten minutes, and had it come back at 0.75 the entire
validation scheme would have needed rethinking before anything else was worth doing.

## The model, and what actually moved it

LightGBM, native categorical handling, native NaN handling, five-fold stratified CV
throughout.

| change | CV | gain |
|---|---|---|
| untuned defaults, 100 trees | 0.954947 | anchor |
| 1000 trees | 0.962141 | +0.007194 |
| lr 0.05, 2000 trees | 0.963210 | +0.001069 |
| bagging 0.8/0.8, 5 seeds, rank averaged | 0.963880 | +0.000670 |
| target and frequency encoding, all 12 columns | 0.966782 | +0.002902 |
| logistic stack over 18 models | 0.967665 | +0.000883 |

The first row is the embarrassing one and it is the most useful. The untuned baseline
was **badly underfit**, and 80% of everything I gained in the first three weeks was
recovered by asking for more trees. Fit the capacity before doing anything clever.

Tuning stopped after the learning rate. The step from lr 0.05 to 0.03 was +0.000064
at 60% more runtime, which is nothing, and the curve had visibly flattened.

The fifth row is a different kind of change from the four above it and it gets its
own section below.

In [ ]:
# Ledger row 9, retrained here so a row reproduces in front of you rather than
# being asserted. Not the final model: it is the raw-feature configuration, chosen
# because it is the one every comparison in this notebook is measured against. This is the slow cell: five folds of
# 2000 trees over 691k rows. Measured at 4.9 minutes on a Kaggle CPU kernel,
# about 56 seconds per fold; slower on fewer cores.
# Set to False to skip and just read the recorded numbers.
RUN_TRAINING = True

RECORDED_SEED42_CV = 0.963471   # experiments.csv row 9

# Do not expect this to reproduce exactly, and do not read a small difference as a bug.
# LightGBM's `deterministic=True` pins the result for a GIVEN THREAD COUNT, not across
# thread counts, because the flag fixes the order in which per-thread gradient sums are
# reduced and that order depends on how many threads there are. Measured on one fold of
# this exact config on the machine the ledger was produced on: n_jobs=6 reproduces the
# recorded value to 3.6e-7, while n_jobs=-1 (8 threads there) lands 3.0e-5 away, and
# both repeat to ~3e-7 across runs. Kaggle's core count is a third thing again.
#
# 1e-4 is the honest bar. It is well below the smallest gain in the ledger (+0.000238)
# and far below the +0.0089 the whole competition was worth, so nothing here rests on
# the fourth decimal of a single run.
TOLERANCE = 1e-4

PARAMS = dict(
    objective="binary", metric="auc", learning_rate=0.05, n_estimators=2000,
    subsample=0.8, subsample_freq=1, colsample_bytree=0.8,
    random_state=42, n_jobs=-1, verbose=-1,
    # Determinism. Without these two, two runs of the same config differ in the
    # fourth decimal, which is the same size as most of the gains being chased.
    deterministic=True, force_row_wise=True,
)

if RUN_TRAINING:
    import lightgbm as lgb
    from sklearn.metrics import roc_auc_score
    from sklearn.model_selection import StratifiedKFold

    from pathlib import Path

    def find_train():
        kag = Path("/kaggle/input")
        # Recursive: Kaggle mounts competitions under
        # /kaggle/input/competitions/<slug>/, not /kaggle/input/<slug>/.
        if kag.exists():
            hits = sorted(kag.rglob("train.csv"))
            if hits:
                return hits[0]
        for b in [Path.cwd(), *Path.cwd().parents]:
            p = b / "data" / "raw" / "train.csv"
            if p.exists():
                return p
        raise FileNotFoundError("train.csv not found")

    train = pd.read_csv(find_train())
    TARGET = "addicted_label"
    CAT = ["gender", "stress_level", "academic_work_impact"]
    feats = [c for c in train.columns if c not in ("id", TARGET)]
    X = train[feats].copy()
    for c in CAT:
        X[c] = X[c].astype("category")
    y = train[TARGET].to_numpy()

    skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
    scores = []
    for f, (tr, va) in enumerate(skf.split(X, y)):
        m = lgb.LGBMClassifier(**PARAMS)
        m.fit(X.iloc[tr], y[tr])
        p = m.predict_proba(X.iloc[va])[:, 1]
        scores.append(roc_auc_score(y[va], p))
        print(f"  fold {f}: {scores[-1]:.6f}")

    cv = float(np.mean(scores))
    diff = cv - RECORDED_SEED42_CV
    print(f"\nCV {cv:.6f} +/- {np.std(scores):.6f}")
    print(f"recorded in the ledger: {RECORDED_SEED42_CV:.6f}")
    print(f"difference: {diff:+.2e}  (expected: within {TOLERANCE:.0e})")
    print("reproduced" if abs(diff) < TOLERANCE else
          "OUTSIDE TOLERANCE - that would be worth investigating")
else:
    print(f"skipped. recorded CV for this config: {RECORDED_SEED42_CV:.6f}")

## Seed averaging, and where it stops paying

With `subsample=0.8` the model becomes stochastic, so the same config under a
different seed is a genuinely different model. Rank-averaging several of them is the
cheapest ensembling there is: no new ideas, no new features, just the same run again.

One warning first. **A single bagged run does not establish that bagging helps.**
Across five seeds the gain over the identical unbagged config was +0.000261,
+0.000024, +0.000235, +0.000127 and +0.000273. One seed got essentially nothing and
one finished *below* the unbagged model. Had I run one seed and read the result, I
would have believed whichever number I happened to draw. Only the blend is solid.

In [ ]:
sat = D["seed_saturation"]
n = [s["n_seeds"] for s in sat]
v = [s["cv"] for s in sat]

fig, ax = plt.subplots(figsize=(7.5, 4.3))
ax.plot(n, v, color=BLUE, lw=2, marker="o", ms=8, mec=SURFACE, mew=1.5, zorder=3)
for xi, yi in zip(n, v):
    ax.annotate(f"{yi:.6f}", (xi, yi), textcoords="offset points", xytext=(0, 11),
                ha="center", fontsize=9, color=SECOND)
for i in range(1, len(n)):
    ax.annotate(f"+{(v[i] - v[i-1]) * 1e6:.0f}e-6",
                ((n[i] + n[i-1]) / 2, (v[i] + v[i-1]) / 2),
                textcoords="offset points", xytext=(0, -18), ha="center",
                fontsize=8.5, color=MUTED)
ax.set_xticks(n)
ax.set_xlabel("seeds in the rank-average blend")
ax.set_ylabel("cross-validation ROC AUC")
ax.set_ylim(min(v) - 0.00008, max(v) + 0.00012)
finish(ax, "Seed averaging saturates at four",
       "Each step is roughly half the last.")
plt.tight_layout()
plt.show()

Each step is about half the one before. Seeds six and seven would together buy
roughly 0.00003, so I stopped at five.

One negative result worth recording: **seed averaging did not reduce fold spread**,
0.000555 against 0.000549 for the best single model. The variance-reduction argument
for ensembling failed to show up three separate times in this competition. I have
stopped repeating it.

## The finding I did not expect: correlation does not predict blend value

The standard advice for ensembling is to look for models that are individually decent
and *uncorrelated*, because decorrelated errors cancel. I gated my whole diversity
plan on that, with thresholds written into a notebook before it ran.

So I measured it. Every pair of the twelve models I had saved out-of-fold predictions
for, 66 pairs, scored as a **50/50 rank blend**, against the Spearman correlation
between the two members. Gain is measured against **the better of the two members**,
which is the only comparison that answers "was blending worth it".

That "50/50" is doing more work than I realised at the time. Read this section and
the next one as measured, and then read the correction two sections down, because the
qualifier turns out to be the whole story.

In [ ]:
pairs = D["pairs"]
BAGGED = {"bagged seed 42", "bagged seed 2024", "bagged seed 7",
          "bagged seed 2025", "bagged seed 13"}
# The clean subset: pairs where the two members share an identical configuration
# and differ only in the random seed. Nothing varies but the stochasticity, so
# nothing is confounded with correlation.
homog = [p for p in pairs if p["a"] in BAGGED and p["b"] in BAGGED]
rest = [p for p in pairs if not (p["a"] in BAGGED and p["b"] in BAGGED)]

r = lambda xs, ys: float(np.corrcoef(xs, ys)[0, 1])
r_all = r([p["spearman"] for p in pairs], [p["gain"] for p in pairs])
r_hom = r([p["spearman"] for p in homog], [p["gain"] for p in homog])
r_gap = r([p["cv_gap"] for p in pairs], [p["gain"] for p in pairs])

fig, axes = plt.subplots(1, 2, figsize=(11.5, 4.9))

ax = axes[0]
ax.axhline(0, color=AXIS, lw=1.2, zorder=1)
ax.plot([p["spearman"] for p in rest], [p["gain"] for p in rest], "o",
        color=MUTED, ms=6, alpha=0.55, lw=0, zorder=2, label="all other pairs")
ax.plot([p["spearman"] for p in homog], [p["gain"] for p in homog], "o",
        color=BLUE, ms=8, mec=SURFACE, mew=1.5, lw=0, zorder=3,
        label="identical config, seed only")
ax.set_xlabel("Spearman correlation between the two models")
ax.set_ylabel("blend gain over the better member")
ax.legend(frameon=False, loc="lower left", labelcolor=SECOND, fontsize=9)
finish(ax, "Correlation does not predict blend value",
       f"all 66 pairs r = {r_all:+.2f}   ·   the 10 clean pairs r = {r_hom:+.2f}")

ax = axes[1]
ax.axhline(0, color=AXIS, lw=1.2, zorder=1)
ax.plot([p["cv_gap"] for p in pairs], [p["gain"] for p in pairs], "o",
        color=BLUE, ms=6.5, mec=SURFACE, mew=1, alpha=0.9, lw=0, zorder=2)
ax.set_xlabel("difference in CV between the two models")
ax.set_ylabel("blend gain over the better member")
finish(ax, "Relative strength does, partly by definition",
       f"r = {r_gap:+.2f}, and a fixed 50/50 weight forces much of it")
plt.tight_layout()
plt.show()

print(f"spearman vs gain, all 66 pairs        : {r_all:+.3f}")
print(f"spearman vs gain, 10 seed-only pairs  : {r_hom:+.3f}")
print(f"cv gap vs gain,   all 66 pairs        : {r_gap:+.3f}")

**Left panel: correlation tells you essentially nothing.** Across all 66 pairs the
relationship between Spearman and blend gain is r = +0.14. The single best blend I
found came from one of the *most* correlated pairs.

**Right panel needs a caveat, and it is important.** Relative strength tracks blend
gain at r = -0.99, but a good part of that is definitional rather than discovered:
at a fixed 50/50 weight, averaging in a much weaker model *has* to drag the result
down. Read the right panel as a constraint the arithmetic imposes, not as a finding.

The honest test of the correlation claim is the blue points, ten pairs of models
with **identical configuration differing only in the random seed**, where nothing is
confounded with correlation. There, Spearman ranges over 0.990 to 0.996, blend gain
ranges over +0.000225 to +0.000360, and the relationship between them is **r =
+0.32 on ten points**, which is noise, and pointing the wrong way for the folk rule.

I want to be careful about what this does and does not license. It is one synthetic
tabular dataset, and every pair above is LightGBM against LightGBM. It is not a
claim that decorrelation is useless in general. What it does establish, for this
data, is that **Spearman was not a usable gate**, and the practical consequence is
concrete:

> CatBoost correlated with my LightGBM at 0.9877, squarely inside the 0.974 to
> 0.998 band that LightGBM produces against *itself*, and blended **negatively**, at
> -0.000250. The bagged seeds correlate *higher*, at 0.990 to 0.996, and blend
> **positively**, at +0.000546.

My original notebook would have read CatBoost's 0.9877 as "marginal diversity,
proceed" and spent the next day on a five-fold CatBoost run at nine times
LightGBM's cost, followed by XGBoost. Measuring the blend directly cost one fold and
stopped that.

**Gate on the measured blend AUC. It is one number and you can always afford it.**

That last line is right as far as it goes, and it is not far enough. Two sections
down it turns out that "the measured blend AUC" is not one number: it is one number
per combiner, and the answer changes sign depending on which one you pick.

## The one test that could have falsified it

Everything above is LightGBM against LightGBM, and that is a real weakness in
the argument. The 66 pairs span Spearman 0.975 to 0.998, and a folk rule about
decorrelation is not really on trial inside a band that narrow. CatBoost, the
one other family I had tried, landed at 0.9877, squarely inside it.

So I ran a genuinely different model: an embedding MLP,
512/256/128, quantile-transformed numeric inputs with an explicit missingness
mask, trained on a Kaggle T4 because the machine I work on has no GPU. Its
out-of-fold vector is the only one in this repo whose correlation with the
LightGBM blend falls **below** the within-family band.

If decorrelation pays, this is where it pays.

The fold split was checksummed on Kaggle against the local one before any of
this was blended. A misaligned out-of-fold vector blends perfectly cleanly and
is undetectable afterwards, which makes it the one error here worth a
dedicated check.


In [ ]:
nu = D["neural"]
curve = nu["weight_curve"]
lo, hi = nu["within_family_spearman_min"], nu["within_family_spearman_max"]

fig, axes = plt.subplots(1, 2, figsize=(11.5, 4.9))

ax = axes[0]
ax.axvspan(lo, hi, color=BLUE, alpha=0.13, zorder=1)
ax.plot([p["spearman"] for p in D["pairs"]],
        np.full(len(D["pairs"]), 1.0), "|", color=BLUE, ms=18, mew=1.2,
        alpha=0.7, zorder=3)
ax.plot([nu["catboost_spearman"]], [1.0], "o", color=MUTED, ms=9,
        mec=SURFACE, mew=1.5, zorder=4)
ax.plot([nu["spearman_vs_blend"]], [1.0], "o", color=ORANGE, ms=10,
        mec=SURFACE, mew=1.5, zorder=5)
ax.annotate("the 66 within-family pairs", ((lo + hi) / 2, 1.0),
            textcoords="offset points", xytext=(0, 30), ha="center",
            fontsize=9, color=SECOND)
ax.annotate("CatBoost", (nu["catboost_spearman"], 1.0),
            textcoords="offset points", xytext=(0, -32), ha="center",
            fontsize=9, color=SECOND)
ax.annotate("neural", (nu["spearman_vs_blend"], 1.0),
            textcoords="offset points", xytext=(0, -32), ha="center",
            fontsize=9.5, color=ORANGE, fontweight="bold")
ax.set_ylim(0.5, 1.5)
ax.set_yticks([])
ax.grid(axis="y", visible=False)
ax.set_xlabel("Spearman correlation with the LightGBM blend")
finish(ax, "The only model that escapes the band",
       f"Within-family pairs run {lo:.4f} to {hi:.4f}. "
       f"The neural model sits at {nu['spearman_vs_blend']:.4f}.")

ax = axes[1]
ax.axhline(0, color=AXIS, lw=1.2, zorder=1)
ax.plot([c["w"] for c in curve], [c["gain"] for c in curve], color=ORANGE,
        lw=2, marker="o", ms=6, mec=SURFACE, mew=1.5, zorder=3)
ax.set_xlabel("weight given to the neural model")
ax.set_ylabel("blend gain over LightGBM alone")
won = sum(c["folds_won"] for c in curve)
finish(ax, "and the worst blend partner in the repo",
       f"Monotone down from the smallest weight tried. "
       f"{won} folds won out of {5 * len(curve)}.")
plt.tight_layout()
plt.show()

best = max(curve, key=lambda c: c["gain"])
print(f"neural CV         {nu['cv']:.6f} +/- {nu['sd']:.6f}")
print(f"LightGBM blend    {nu['lgbm_blend_cv']:.6f}")
print(f"behind by         {nu['cv'] - nu['lgbm_blend_cv']:+.6f}")
print(f"best weight w={best['w']:.2f}  gain {best['gain']:+.6f}, "
      f"wins {best['folds_won']}/5 folds")


**Spearman 0.9650, below every one of the 66 within-family pairs, and it is the worst
blend partner I measured.** The curve is monotone downward from the smallest weight I
tried. Zero folds won, at any weight. Nothing about it is marginal or close.

Every number in that chart is correct and it reproduces from the saved vector. What I
concluded from it was not. At the time I wrote: the decorrelated model did not help,
the near-identical bagged seeds did, and what predicts blend value on this data is not
how *different* two models are but whether they are **comparably strong**. Then I
wrote "the board is now closed" into my working notes and started writing this
notebook up.

## The one feature idea that worked, and it is not a feature

That closed board lasted six days. Two things came after it, and they are the two
largest results in the competition.

Three weeks of feature engineering on this dataset had produced nothing. Missingness
indicators: dead in ninety seconds, above. Threshold and rule features: below the
untuned baseline. Ratios and interactions: nothing. With twelve columns and a
generator that writes a smooth additive field, there was no hidden quantity to
construct.

Target encoding is not a new quantity. It replaces each column with the mean target
rate of the rows sharing its value, so a categorical or a discretised numeric arrives
as one number a tree can split on in a single cut, where before it cost many. Applied
to all twelve columns alongside a frequency encoding, 12 features become 36:

| | CV | fold sd |
|---|---|---|
| the same model, raw features | 0.963471 | 0.000591 |
| the 5-seed blend it replaced | 0.963880 | 0.000555 |
| **target encoded** | **0.966782** | **0.000453** |

**+0.003312 against the identical model on raw features, winning 5 folds out of 5,
with the per-fold differences having a standard deviation of 0.000270.** That is
twelve paired standard deviations. For scale, the entire seed-averaging programme
above was worth about one.

**The credit is not mine.** The idea came from a public notebook by tomasa2 on this
competition. The implementation, the leak checks and the cross-validation are mine,
which is the part worth writing down, because target encoding is the one thing in
this repo that can leak.

**How it is nested, and how I know it did not leak.** Inside each of the five outer
folds, an inner 5-fold split fits the encoder on four inner parts and writes the
encoded values for the fifth, so no row's own target ever reaches its own feature.
Smoothing 10 toward the prior. Three checks run inside the training notebook on the
full data rather than a sample:

- a validation row's encoded value against its own target: correlation **0.0**, since
  the encoder that produced it never saw that fold;
- a training row's encoded value against its own target: **8.1e-05**, below the
  **1.3e-04** that the leave-one-out arithmetic produces on its own;
- an encoder deliberately fit *without* the nesting, on the same folds: **0.9967**.

That third one is the check that matters. A leak here is worth three points of AUC,
so a broken implementation is loud. Reporting the two clean numbers without the third
would be reporting that an alarm did not go off without checking it was wired in.

### Two things that failed on top of it, both worth more than they cost

**Median-imputed copies of the nine numeric columns, added alongside the originals.**
A public notebook reported +0.0012 for this. I measured **+0.000023**, paired sd
0.000032. It won 4 folds of 5, which sounds positive until you notice the magnitude
is 0.7 paired standard deviations against target encoding's twelve. Fold count with no
magnitude behind it is not evidence. My guess, logged as a guess: they measured it
against a baseline without full target encoding, and once every column is encoded the
NaN level already receives its own value, so the imputed copies are redundant.

**The decimal lattice, and this is the one I would take away.** The float columns in
this generator are not uniform in their last digit, and the target rate depends on it.
Measured on the full training set, roughly 50,000 to 70,000 rows behind each digit:

| column | swing in target rate across the first decimal digit |
|---|---|
| `weekend_screen_time` | 11.54 points |
| `daily_screen_time_hours` | 8.90 points |
| `sleep_hours` | 7.12 points |
| `social_media_hours` | 5.53 points |

`daily_screen_time_hours` runs from 0.6482 at digit `.0` to 0.7373 at digit `.9`.
That is roughly 47 standard errors. The effect is real, it is enormous, and I verified
it here rather than taking it from the forum.

Adding the digit as a feature scored **-0.000132**, winning 1 fold of 5.

The reason is worth the paragraph. The digit is a **deterministic function of the
value**, so target encoding the value already carries whatever the digit contributes
to it. What a separate digit column adds is *pooling* across integer parts, and
pooling only pays when the per-value estimate is noisy. At about 500 rows per encoded
level, it is not. A large verified effect in the data can be worth exactly zero in
the model, and those are two different measurements.

## The correction: that was a fact about my combiner

The second thing that came after the closed board is a correction to the section
before last, and it is the one I would keep out of this whole notebook.

Every ensembling result above shares a hidden constant. The 66 pairs, the CatBoost
probe, the seven-way blend, the neural weight curve: **every one of them combined
models by averaging them.** Rank average, or a fixed share of one against the other.

A combiner with no weights can only average a member in. It cannot give it a small
weight and it cannot give it a negative one. So the experiment I never ran is the one
that holds the member set fixed and changes only the combiner.

Eighteen out-of-fold vectors: five target-encoded seeds, the twelve raw-feature
LightGBMs, and the neural model. Every stacker below is **fit on half the out-of-fold
rows and scored on the other half**, five random splits, paired. A stacker fitted and
scored on the same matrix reads high, and that optimism is exactly what would
manufacture the result I am claiming here, so it has to be measured out.

In [ ]:
comb = {c["name"]: c for c in D["combiners"]}
SHOW = ["rank mean, 5 seeds", "logit stack, 5 seeds",
        "rank mean, all 18", "logit stack, all 18"]

fig, axes = plt.subplots(1, 2, figsize=(11.5, 4.9))

ax = axes[0]
vals = [comb[k]["gain"] for k in SHOW]
yy = np.arange(len(SHOW))
ax.barh(yy, vals, color=[ORANGE if v < 0 else BLUE for v in vals], height=0.62,
        zorder=3)
ax.axvline(0, color=AXIS, lw=1.2, zorder=4)
for i, v in enumerate(vals):
    ax.annotate(f"{v:+.6f}", (v, i), textcoords="offset points",
                xytext=(8 if v > 0 else -8, 0), va="center",
                ha="left" if v > 0 else "right", fontsize=9, color=SECOND)
ax.set_yticks(yy)
ax.set_yticklabels(SHOW, fontsize=9.5, color=SECOND)
ax.invert_yaxis()
ax.set_xlim(min(vals) * 1.55, max(vals) * 1.9)
ax.set_xlabel("gain over the best single model")
ax.grid(axis="y", visible=False)
finish(ax, "The combiner, not the members",
       "Bottom two rows are the same eighteen models. Averaging them loses;\n"
       "letting a logistic regression choose the weights wins.")

# Coefficients against each member's own CV. The claim in the title is that these two
# quantities come apart, so both have to be measured rather than asserted.
ax = axes[1]
co = D["stack_coefs"]
nn = [c for c in co if c["name"] == "neural"]
rest = [c for c in co if c["name"] != "neural"]
ax.axhline(0, color=AXIS, lw=1.2, zorder=1)
ax.plot([c["cv"] for c in rest], [c["coef"] for c in rest], "o", color=MUTED, ms=7,
        alpha=0.75, lw=0, zorder=2, label="LightGBM, 17 of them")
ax.plot([c["cv"] for c in nn], [c["coef"] for c in nn], "o", color=ORANGE, ms=10,
        mec=SURFACE, mew=1.5, lw=0, zorder=3, label="the neural model")
ax.annotate("rejected on the evidence\nin the section above",
            (nn[0]["cv"], nn[0]["coef"]), textcoords="offset points", xytext=(12, -6),
            fontsize=9, color=SECOND, linespacing=1.35)
ax.set_xlabel("the member's own cross-validation AUC")
ax.set_ylabel("its weight in the stack")
ax.legend(frameon=False, loc="lower right", labelcolor=SECOND, fontsize=9)
finish(ax, "Weight is not strength",
       "The weakest member of the eighteen takes the seventh largest weight.\n"
       "The negative weights are underfit models used as corrections.")
plt.tight_layout()
plt.show()

for k in SHOW:
    c = comb[k]
    print(f"{k:22} {c['auc']:.6f}  {c['gain']:+.6f}  "
          f"sd {c['paired_sd']:.6f}  {c['splits_won']}/5 splits")

**Same eighteen models. Opposite sign.** Averaged, they lose 0.001554 and win zero
splits out of five. Weighted, they gain 0.000908 and win all five, with a paired
standard deviation of 0.000018. The distance between those two bars is comparable to
the entire target-encoding result, and none of it is attributable to the models.

Both of those are split-half numbers. Refitting the combiner inside the five-fold loop
instead, which is how every other number in my ledger is produced, gives **+0.000867
on five folds out of five**. The result does not depend on which of the two honest
protocols measures it, and the optimism in the naive version, fitting and scoring the
stacker on the same rows, turned out to be 1.5e-05: eighteen parameters on 691,369
rows is not enough to overfit with.

The two upper bars are the other half of the story. Among five seeds of one
configuration, the rank average and the fitted stack agree to five decimals. When
members are equally strong and near identical, a stacker has nothing to do that an
average is not already doing. **That is the regime every ensembling experiment in this
repo was run in**, which is why those results were trustworthy and why the conclusion
I drew from them was not.

The neural model is the sharpest case, since it is the one I closed the board on:

| paired with the best single model | gain | splits won |
|---|---|---|
| rank average | -0.006272 | 0/5 |
| logistic stack | **+0.000090** | **5/5** |

Same two vectors, same rows. Averaged it is a disaster and the weight curve above was
right. Weighted it is positive on every split. Dropping it from the eighteen and
refitting the other seventeen costs **0.000093**, paired sd 0.000003, five splits out
of five: a tenth of the stack's total gain, from the model I had called the worst
partner I measured.

The right panel says why. `anchor` and `300 trees` are underfit LightGBMs whose errors
are systematic, so the stacker takes them at **-0.36** and **-0.13** and uses them as
corrections. `bagged seed 2024` is a strong model nearly identical to four others in
the set and gets **+0.01**, because it is redundant. The neural model gets **+0.12**
because it is the only member wrong in a different direction.

**What survives, stated carefully.** Rank correlation still does not predict
equal-weight blend value; the 66-pair result is untouched, and so is the CatBoost
decision, which was made on cost as well. Averaging in a much weaker model still
drags the average down, which is arithmetic. What does not survive is the
generalisation I made from those: that a weak decorrelated model has no value on this
data. It has value the moment the combiner is allowed to weight it.

**How I missed it for a week.** I had already caught myself once here. The original
gate was the Spearman threshold, I measured that it was anti-predictive, and I
replaced it with "measure the blend AUC directly". That was the right correction and
it fixed the statistic. It left the combiner exactly where it was, and by then the
answer looked known, so nothing asked the next question. The cost was one rejected
model, three days of a closed board, and most of a writeup arguing the wrong
conclusion at length.

**The rule I would give someone else:** a diversity claim is a claim about a member
set *and* a combiner. Put both in the sentence, or you will generalise one of them by
accident.

## Fold standard deviation is the wrong bar for a paired comparison

A rule I had written down for myself, and which is good general advice: do not
believe an improvement smaller than the fold-to-fold spread.

By that rule my one real ensembling result was inconclusive. The seed blend gained
**+0.000546** over the previous best against a fold spread of **0.000549**.

The rule is wrong here, and it is worth understanding why. The per-fold differences
were:

```
+0.000588  +0.000631  +0.000476  +0.000620  +0.000419
```

Five folds out of five, with a standard deviation across those differences of
**0.000094**, a sixth of the fold spread.

Fold-to-fold variation is mostly driven by *which rows landed in which fold*. That
variation is common to both models being compared, so it **cancels in the
difference**. Comparing a paired improvement against the unpaired fold spread throws
away the pairing and inflates your bar by roughly six times.

**The right bar for "is model B better than model A" is the spread of the per-fold
differences, plus how many folds it wins.** By that bar the result is unambiguous.
It nearly got discarded.

## What CV and the public leaderboard actually did

In [ ]:
# Row 16 is the neural model at 0.9392. Different family, never submitted, and
# plotting it here would flatten every other row into a line. It has its own figure
# above rather than being dropped in silence, and the title below says so.
led = [r for r in D["ledger"] if r["id"] != 16]
x = [r["id"] for r in led]
cv = [r["cv_mean"] for r in led]
by_id = {r["id"]: r["cv_mean"] for r in led}
# `is not None` matters: an unsubmitted row carries a null here, and a bare
# truthiness test would let it through and rely on matplotlib silently dropping it.
lb_x = [r["id"] for r in led if r["lb_public"] is not None]
lb_y = [r["lb_public"] for r in led if r["lb_public"] is not None]

fig, ax = plt.subplots(figsize=(9.5, 4.6))
ax.plot(x, cv, color=BLUE, lw=2, marker="o", ms=5.5, label="cross-validation",
        zorder=3, mec=SURFACE, mew=1.5)
ax.plot(lb_x, lb_y, color=ORANGE, lw=0, marker="D", ms=7, mec=SURFACE, mew=1.5,
        label="public leaderboard", zorder=4)
for xi, yi in zip(lb_x, lb_y):
    ax.annotate(f"{yi:.5f}", (xi, yi), textcoords="offset points", xytext=(0, 9),
                ha="center", fontsize=8.5, color=SECOND)
ax.annotate("untuned anchor", (1, by_id[1]), textcoords="offset points",
            xytext=(10, 7), fontsize=9, color=SECOND)
ax.annotate("target encoding", (17, by_id[17]), textcoords="offset points",
            xytext=(4, -22), fontsize=9, color=SECOND)
ax.set_xlabel("experiment")
ax.set_ylabel("ROC AUC")
ax.set_xticks(x)
ax.legend(frameon=False, loc="lower right", labelcolor=SECOND)
finish(ax, "Every experiment in the ledger",
       "Fifteen experiments of capacity and averaging, worth +0.0089 together.\n"
       "Then one change of representation, worth +0.0029 on its own.")
plt.tight_layout()
plt.show()

The leaderboard sits above CV by between 0.0010 and 0.0022 on all seven submissions
and has never once ranked two of them in the wrong direction. That is the whole job
of a validation scheme, and rows 17 and 24 are the strongest version of the check
available here, because they are the only two submissions that changed the feature
set and the combiner rather than the hyperparameters. CV moved +0.0029 and +0.0009;
the leaderboard moved +0.0032 and +0.0007. The stack was predicted at 0.9691 from the
running offset before it was submitted and came back at 0.96897.

The interesting part is the two submissions before that. My 3-seed and 5-seed blends
are separated **cleanly** by CV: the 5-seed wins 5 folds out of 5 with a paired
standard deviation of 0.000023. On the public leaderboard they scored **0.965090** and
**0.965080**, a gap of 0.00001 in the other direction.

That is not a CV/LB disagreement to diagnose. The public leaderboard here is a sample
of roughly 89,000 rows and its standard error is near **0.001**, larger than every
single gain after the fourth experiment. It is a 6e-5 effect measured with a 1e-3
ruler.

The practical consequence for final submission selection: the usual split is best-CV
plus best-public-LB, and when the public LB's noise floor exceeds the effects you are
choosing between, **best-public-LB is close to picking at random**. CV gets the
heavier weight here.

## The ledger, and what I would do differently

Twenty-four experiments, every one written down including the failures. The failures
are most of the value: they are why the same dead end did not get walked twice, and
four of them are in this notebook at the same length as the wins.

**What worked**

- Fitting model capacity first. 80% of the gain from the first three weeks.
- Lower learning rate with a compensating tree count. A small gain, and a useful side
  effect: fold spread fell from 0.000816 to 0.000549, so every later comparison got
  easier to call.
- Bagged seed averaging, rank-blended, up to four seeds.
- Target and frequency encoding on all twelve columns, +0.0029, nested inside the fold
  loop and leak-checked three ways.
- A logistic stack over everything trained, +0.0009 on top of that, including the
  models that were individually rejected.

**What did not**

- Missingness indicators. No signal, established in ninety seconds.
- Threshold and rule features. The generator caps them below the untuned baseline.
- External data. The original is 7,500 rows against 691,369, its distributions are
  warped relative to the synthetic, and it appears to be synthetic itself.
- CatBoost. 0.0017 behind at matched budget, blends negatively, nine times the cost.
- Capacity diversity within LightGBM, under an equal-weight blend.
- Median-imputed columns on top of target encoding. +0.000023, a null.
- The decimal lattice. An 11-point effect in the data, -0.000132 in the model.

**What I got wrong and had to publish a correction to**

- The neural model, and with it the whole diversity conclusion. It is 0.0247 behind
  and it makes an *average* worse at every weight, which is what I measured and what I
  reported. What I then claimed was that diversity does not pay on this data, and that
  was a fact about equal-weight combiners rather than about the data. It takes a
  positive weight in the stack, and dropping it costs a tenth of the stack's gain.

**What I would do differently**

1. **Time one fold before launching anything.** I started a CatBoost run estimated at
   30-50 minutes. It ran for two hours with `verbose=0` and no progress output before
   being killed. It was never hung: a single fold was ten minutes and the notebook
   called the pipeline twice. The estimate was wrong by 3x because no fold had been
   timed.
2. **Name the combiner in any claim about ensembling.** See above. This is the
   expensive one.
3. **Not offer a mechanism from one data point.** I attributed the CV/LB gap to fold
   averaging on the first submission, patched the story on the second, and withdrew it
   on the third when the pattern did not hold.
4. **Check the row count before calling something a lever.** I described the external
   dataset as one of the few real levers available, then found it was a 1% increase in
   training data.
5. **Not declare a board closed.** I wrote that sentence with eight rejected ideas
   behind it, and the two largest gains in the competition came after it.

**What I would keep**

The checksummed fold split. It is ten lines, and it is the reason an out-of-fold
vector saved in week one could be stacked against one from week four with any
confidence at all. Half of this notebook would be unmeasurable without it.

---

Ledger, notebooks and working log: **github.com/vyask21/smartphone-addiction**

If one thing here is worth taking away, it is that a cheap test of a wrong idea is
worth more than an expensive test of a good one, and most ideas are wrong. Including,
twice in this notebook, mine.